# Custom Tirex CPM Evaluation

In [1]:
import sys
sys.path.append('../tirex/src') # Add the path to the tirex module

In [23]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
import torch
import json
import altair as alt
from datetime import datetime
from pathlib import Path

from typing import Dict, Optional

from tirex import ForecastModel, load_model
from tirex_loss.models.base_model import Base_Model
from tirex_loss.logger import TrainingLogger
from tirex_loss.logger.plot import plot_training_curves

# set default figure size for all plots
plt.rcParams["figure.figsize"] = (12, 6)
# set default seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load data

Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [24]:
data_path_train = "../data/train.csv"
data_path_val = "../data/validation.csv"
data_path_test = "../data/test.csv"

data_train = pl.read_csv(data_path_train, try_parse_dates=True)
data_val = pl.read_csv(data_path_val, try_parse_dates=True)
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series_train = len(data_train['series_index'].unique())
number_series_val = len(data_val['series_index'].unique())
number_series_test = len(data_test['series_index'].unique())

print(f"Number of time series in the train set: {number_series_train} | Length of the train set: {len(data_train)}")
print(f"Number of time series in the val set: {number_series_val} | Length of the val set: {len(data_val)}")
print(f"Number of time series in the test set: {number_series_test} | Length of the test set: {len(data_test)}")
data_test.head()

Number of time series in the train set: 361 | Length of the train set: 17732900
Number of time series in the val set: 361 | Length of the val set: 3799959
Number of time series in the test set: 361 | Length of the test set: 3800086


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [25]:
# reduce amount of data
amount_train = 0.20
amount_val = 0.20
amount_test = 0.20

series_idx_train = data_train['series_index'].unique()
number_series_train = len(series_idx_train)
number_series_train_final = int(number_series_train * amount_train)
data_train_sampled = data_train.filter(pl.col('series_index').is_in(series_idx_train.sample(number_series_train_final).implode()))

series_idx_val = data_val['series_index'].unique()
number_series_val = len(series_idx_val)
number_series_val_final = int(number_series_val * amount_val)
data_val_sampled = data_val.filter(pl.col('series_index').is_in(series_idx_val.sample(number_series_val_final).implode()))

series_idx_test = data_test['series_index'].unique()
number_series_test = len(series_idx_test)
number_series_test_final = int(number_series_test * amount_test)
data_test_sampled = data_test.filter(pl.col('series_index').is_in(series_idx_test.sample(number_series_test_final).implode()))

print(f"Training on {number_series_train_final} out of {number_series_train} series.")
print(f"Validating on {number_series_val_final} out of {number_series_val} series.")
print(f"Testing on {number_series_test_final} out of {number_series_test} series.")

Training on 72 out of 361 series.
Validating on 72 out of 361 series.
Testing on 72 out of 361 series.


# Load Model

In [26]:
# use custom model
model = Base_Model(context_length=1024,
                   quantiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                   patch_size=32,
                   use_slstm=True)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model device: {device}")

Total Parameters: 1,610,816
Model device: cuda


In [27]:
# for restoring the original behavior of the model after our modifications
from tirex import TiRexZero

TiRexZero._original_forecast_tensor = TiRexZero._forecast_tensor

def restore_original_behavior(print_message: bool = True):
    if hasattr(TiRexZero, '_original_forecast_tensor'):
        TiRexZero._forecast_tensor = TiRexZero._original_forecast_tensor
        if print_message:
            print("Restored original NaN-filling behavior.")
    else:
        print("Original method backup not found.")

# Dataloader

Dataloader produces sequences of length $n$ for the input and target sequences with random predictions lengths in the range of $s_{\min}$ and $s_{\max}$.

In [28]:
from tirex_loss.dataloader import (build_dataloader,
                                   build_dataloader_fixed,
                                   build_dataloader_from_dataset,
                                   TirexDataset, TirexDataset_Fixed)
from tirex_loss.dataloader.utils import create_windows, create_windows_fixed

In [29]:
n = model.context_length
s_min = 32
s_max = int(model.context_length / 2)
batch_size = 32

dataloader_train = build_dataloader(data_train_sampled,
                                    context_length=n,
                                    s_min=s_min,
                                    s_max=s_max,
                                    batch_size=batch_size,
                                    shuffle=True,
                                    num_workers=0,
                                    pin_memory=True
                                    )
dataloader_val = build_dataloader(data_val_sampled,
                                  context_length=n,
                                  s_min=s_min,
                                  s_max=s_max,
                                  batch_size=batch_size,
                                  shuffle=False,
                                   num_workers=0,
                                   pin_memory=True
                                   )
dataloader_test = build_dataloader(data_test_sampled,
                                   context_length=n,
                                   s_min=s_min,
                                   s_max=s_max,
                                   batch_size=batch_size,
                                   shuffle=False,
                                   num_workers=0,
                                   pin_memory=True
                                   )

len(dataloader_train), len(dataloader_val), len(dataloader_test)

(377, 69, 70)

# Evaluation

In [30]:
from tqdm.auto import tqdm
from tirex_loss.modifications.autoregressive import autoregressive_mean_forecast_tensor
from tirex_loss.evaluation.metrics import score_data
from torch.utils.data import DataLoader

model_files = ['best_model.pth', 'last_model.pth']

In [31]:
def compare_autoregressive(dataloader_test: DataLoader, device:str):
    model.eval()
    scores_comparison = {"original": (0, 0, 0, 0, 0), "autoregressive_mean": (0, 0, 0, 0, 0)}
    quantiles_comparison = {"original": [], "autoregressive_mean": []}

    batch_bar = tqdm(dataloader_test, total=len(dataloader_test),
                            leave=False, position=1, desc='Batches')
    for i, (x_batch, y_batch, pred_length_batch) in enumerate(batch_bar):
        # put tensors on same device
        x = x_batch.to(device)
        y = y_batch.to(device)
        pred_length = pred_length_batch[0].item()

        # change shape to (n_samples, n_outputs)
        y_true = y.reshape(-1, 1)
        y_train = x.reshape(-1, 1)

        quantiles_original = model(x,
                                    prediction_length=pred_length,
                                    autoregressive=False
                                    )
        quantiles_comparison["original"].append(quantiles_original.detach().cpu().numpy())

        # scores, output is (batch, quantile, time), need to reshape to (batch*time, 1, quantile) for scoring
        y_pred = quantiles_original.reshape(quantiles_original.shape[0]*quantiles_original.shape[2],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.detach().cpu().numpy(),
                    y_pred=y_pred.detach().cpu().numpy(),
                    y_train=y_train.detach().cpu().numpy(),
                    quantiles=model.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["original"] = tuple(a + b for a, b in zip(scores_comparison["original"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

        # now with autoregressive mean filling
        quantiles_autoregressive = model(x,
                                            prediction_length=pred_length,
                                            autoregressive=True
                                            )
        quantiles_comparison["autoregressive_mean"].append(quantiles_autoregressive.detach().cpu().numpy())

        # scores
        y_pred = quantiles_autoregressive.reshape(quantiles_autoregressive.shape[0]*quantiles_autoregressive.shape[2],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.detach().cpu().numpy(),
                    y_pred=y_pred.detach().cpu().numpy(),
                    y_train=y_train.detach().cpu().numpy(),
                    quantiles=model.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["autoregressive_mean"] = tuple(a + b for a, b in zip(scores_comparison["autoregressive_mean"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

    scores_comparison["original"] = tuple(score / len(dataloader_test) for score in scores_comparison["original"])
    scores_comparison["autoregressive_mean"] = tuple(score / len(dataloader_test) for score in scores_comparison["autoregressive_mean"])    

    return scores_comparison, quantiles_comparison

def save_scores(scores: dict, path: str = "scores.json") -> None:
    with open(path, "w") as f:
        json.dump(scores, f, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else x)

In [32]:
def compare_autoregressive_pretrained(dataloader_test: DataLoader, device:str, run_autoregressive: Optional[bool] = True):
    scores_comparison = {"original": (0, 0, 0, 0, 0), "autoregressive_mean": (0, 0, 0, 0, 0)}
    quantiles_comparison = {"original": [], "autoregressive_mean": []}
    mean_comparison = {"original": [], "autoregressive_mean": []}

    batch_bar = tqdm(dataloader_test, total=len(dataloader_test),
                            leave=False, position=1, desc='Batches')
    for i, (x_batch, y_batch, pred_length_batch) in enumerate(batch_bar):
        # put tensors on same device
        x = x_batch.to(device)
        y = y_batch.to(device)
        pred_length = pred_length_batch[0].item()

        # change shape to (n_samples, n_outputs)
        y_true = y.reshape(-1, 1)
        y_train = x.reshape(-1, 1)

        restore_original_behavior(print_message=False)
        quantiles_original, mean_original = model.forecast(x,
                                                           prediction_length=pred_length,
                                                           output_device=device
                                                           )
        quantiles_comparison["original"].append(quantiles_original.cpu().numpy())
        mean_comparison["original"].append(mean_original.cpu().numpy())

        # scores
        y_pred = quantiles_original.reshape(quantiles_original.shape[0]*quantiles_original.shape[1],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.cpu().numpy(),
                    y_pred=y_pred.cpu().numpy(),
                    y_train=y_train.cpu().numpy(),
                    quantiles=model.config.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["original"] = tuple(a + b for a, b in zip(scores_comparison["original"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

        # now with autoregressive mean filling
        if run_autoregressive:
            TiRexZero._forecast_tensor = autoregressive_mean_forecast_tensor
            quantiles_autoregressive, mean_autoregressive = model.forecast(x,
                                                                        prediction_length=pred_length,
                                                                        output_device=device
                                                                        )
            quantiles_comparison["autoregressive_mean"].append(quantiles_autoregressive.cpu().numpy())
            mean_comparison["autoregressive_mean"].append(mean_autoregressive.cpu().numpy())

            # scores
            y_pred = quantiles_autoregressive.reshape(quantiles_autoregressive.shape[0]*quantiles_autoregressive.shape[1],1,-1)
            score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                        y_true=y_true.cpu().numpy(),
                        y_pred=y_pred.cpu().numpy(),
                        y_train=y_train.cpu().numpy(),
                        quantiles=model.config.quantiles,
                        log_scores=False,
                        multioutput='uniform_average'
                        )

            scores_comparison["autoregressive_mean"] = tuple(a + b for a, b in zip(scores_comparison["autoregressive_mean"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))
        else:
            scores_comparison["autoregressive_mean"] = scores_comparison["original"]

    scores_comparison["original"] = tuple(score / len(dataloader_test) for score in scores_comparison["original"])
    scores_comparison["autoregressive_mean"] = tuple(score / len(dataloader_test) for score in scores_comparison["autoregressive_mean"])    

    return scores_comparison, quantiles_comparison, mean_comparison

## Variants

In [33]:
from tirex_loss.loss import QuantileLoss, LossTypes

In [36]:

training_variants = {
    # quantile loss
    # 'custom_tirex_shift_slstm_quantile': {'Autoregressive': False,
    #                                         'use_slstm': True,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    #                                         'dataloader': 'shift',
    #                                         'model_path': Path("../models/v2_custom_tirex_shift_slstm_quantile_20260801_145452")
    #                                         },
    # 'custom_tirex_shift_lstm_quantile': {'Autoregressive': False,
    #                                         'use_slstm': False,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    #                                         'dataloader': 'shift',
    #                                         'model_path': Path("../models/v2_custom_tirex_shift_lstm_quantile_20260801_180502")
    #                                         },
    # 'custom_tirex_cpm_slstm_quantile': {'Autoregressive': False,
    #                                         'use_slstm': True,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    #                                         'dataloader': 'cpm',
    #                                         'model_path': Path("../models/v2_custom_tirex_cpm_slstm_quantile_20260801_220315")
    #                                         },
    # 'custom_tirex_cpm_lstm_quantile': {'Autoregressive': False,
    #                                         'use_slstm': False,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    #                                         'dataloader': 'cpm',
    #                                         'model_path': Path("../models/v2_custom_tirex_cpm_lstm_quantile_20260802_012058")
    #                                         },

    # mse loss
    # 'custom_tirex_shift_slstm_mse': {'Autoregressive': False,
    #                                         'use_slstm': True,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5],
    #                                         'dataloader': 'shift',
    #                                         'model_path': Path("../models/v2_custom_tirex_shift_slstm_mse_20260802_052019")
    #                                         },
    # 'custom_tirex_shift_lstm_mse': {'Autoregressive': False,
    #                                         'use_slstm': False,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5],
    #                                         'dataloader': 'shift',
    #                                         'model_path': Path("../models/v2_custom_tirex_shift_lstm_mse_20260802_082935")
    #                                         },
    # 'custom_tirex_cpm_slstm_mse': {'Autoregressive': False,
    #                                         'use_slstm': True,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5],
    #                                         'dataloader': 'cpm',
    #                                         'model_path': Path("../models/v2_custom_tirex_cpm_slstm_mse_20260802_160458")
    #                                         },
    # 'custom_tirex_cpm_lstm_mse': {'Autoregressive': False,
    #                                         'use_slstm': False,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5],
    #                                         'dataloader': 'cpm',
    #                                         'model_path': Path("../models/v2_custom_tirex_cpm_lstm_mse_20260802_191727")
    #                                         },

    # l1 loss
    'custom_tirex_shift_slstm_l1': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'dataloader': 'shift',
                                            'model_path': Path("../models/v2_custom_tirex_shift_slstm_l1_20260802_231523")
                                            },
    'custom_tirex_shift_lstm_l1': {'Autoregressive': False,
                                            'use_slstm': False,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'dataloader': 'shift',
                                            'model_path': Path("../models/v2_custom_tirex_shift_lstm_l1_20260803_022651")
                                            },
    'custom_tirex_cpm_slstm_l1': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'dataloader': 'cpm',
                                            'model_path': Path("../models/v2_custom_tirex_cpm_slstm_l1_20260803_062017")
                                            },
    'custom_tirex_cpm_lstm_l1': {'Autoregressive': False,
                                            'use_slstm': False,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'dataloader': 'cpm',
                                            'model_path': Path("../models/v2_custom_tirex_cpm_lstm_l1_20260803_093314")
                                            },
}

In [37]:
context_length = 1024
patch_size = 32

for k, v in training_variants.items():
    model_quantiles = v['quantiles']
    autoregressive = v['Autoregressive']
    use_slstm = v['use_slstm']
    model_path = v['model_path']


    run_dirs = sorted([p for p in model_path.iterdir() if p.is_dir()])

    for run_dir in run_dirs:
        # create a new model
        model = Base_Model(context_length=context_length,
                        quantiles=model_quantiles,
                        patch_size=patch_size,
                        use_slstm=use_slstm)
        model.to(device)

        scores = {}
        quantiles = {}
        for model_file in model_files:
            dict_path = run_dir / model_file
            state_dict = torch.load(dict_path, weights_only=True)
            model.load_state_dict(state_dict)

            scores_comparison, quantiles_comparison = compare_autoregressive(dataloader_test, device)
            scores[model_file] = scores_comparison
            quantiles[model_file] = quantiles_comparison

        save_scores(scores, path=run_dir / "scores.json")           


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

/home/richard/tirex_loss/.venv/lib/python3.12/site-packages/xlstm/blocks/slstm/cell.py:543: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @conditional_decorator(
/home/richard/tirex_loss/.venv/lib/python3.12/site-packages/xlstm/blocks/slstm/cell.py:568: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @conditional_decorator(


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

ValueError: Input contains NaN.

## Pre-trained

In [ ]:
# load model
model_path = Path("../models/v2_tirex_original_pretrained_full") # just to save scores
model: ForecastModel = load_model("NX-AI/TiRex")
model.to(device)

scores = {}
quantiles = {}
for model_file in model_files:

    scores_comparison, quantiles_comparison, mean_comparison = compare_autoregressive_pretrained(dataloader_test, device)
    scores[model_file] = scores_comparison
    quantiles[model_file] = quantiles_comparison

save_scores(scores, path=model_path / "scores.json")

Batches:   0%|          | 0/109 [00:00<?, ?it/s]

Batches:   0%|          | 0/109 [00:00<?, ?it/s]

# Summary

In [ ]:
model_paths = {
    key: value["model_path"]
    for key, value in training_variants.items()
}

## Training

In [ ]:
import statistics

def collect_run_summaries(model_path: Path) -> list[dict]:
    """Call get_summary() once per run subfolder."""
    summaries = []
    for run_dir in discover_runs(model_path):
        logger = TrainingLogger(log_dir=run_dir)
        logger.load_metrics_file()  # populates logger.metrics used by get_summary()
        s = logger.get_summary()
        if s:  # skip empty dicts (e.g. missing epoch_metrics)
            s['run'] = run_dir.name
            summaries.append(s)
    return summaries


def aggregate_summaries(run_summaries: list[dict]) -> dict:
    """Mean/std/min/max across runs for each numeric summary field."""
    if not run_summaries:
        return {}

    keys = [k for k in run_summaries[0].keys() if k != 'run']
    agg = {'n_runs': len(run_summaries)}

    for k in keys:
        values = [s[k] for s in run_summaries if s.get(k) is not None]
        if not values:
            agg[f'{k}_mean'] = None
            agg[f'{k}_std'] = None
            continue
        agg[f'{k}_mean'] = statistics.fmean(values)
        agg[f'{k}_std'] = statistics.stdev(values) if len(values) > 1 else 0.0
        agg[f'{k}_min'] = min(values)
        agg[f'{k}_max'] = max(values)

    return agg


def discover_runs(model_path: Path) -> list[Path]:
    """Return run subfolders if present, otherwise treat model_path itself as one run."""
    subdirs = sorted(d for d in model_path.iterdir() if d.is_dir())
    return subdirs if subdirs else [model_path]

def load_all_runs(model_path: Path) -> pl.DataFrame:
    dfs = []
    for run_dir in discover_runs(model_path):
        logger = TrainingLogger(log_dir=run_dir)
        metrics = logger.load_metrics_file()
        edf = metrics.get('epoch_metrics', None)
        if edf is None:
            continue
        # if 'test_loss' not in edf.columns and 'val_loss' in edf.columns:
        #     edf = edf.rename({'val_loss': 'test_loss'})        
        edf = edf.with_columns(pl.lit(run_dir.name).alias('run'))
        dfs.append(edf)
    if not dfs:
        return None
    return pl.concat(dfs, how='vertical_relaxed')


def compute_shared_y_domains_by_loss(model_paths: dict[str, Path], padding: float = 1.05) -> dict:
    """Compute a shared loss y-domain per loss type, across all models and all runs.

    Returns a dict mapping loss class -> [0, max_value] domain.
    """
    values_by_loss = {}
    for name, p in model_paths.items():
        loss_key = name.rsplit("_", 1)[-1]  # e.g. "original_quantile" -> "quantile"
        values_by_loss.setdefault(loss_key, [])

        for run_dir in discover_runs(p):
            logger = TrainingLogger(log_dir=run_dir)
            metrics = logger.load_metrics_file()
            epoch_df = metrics.get('epoch_metrics', None)
            if epoch_df is None:
                continue
            val_metric = 'test_loss' if 'test_loss' in epoch_df.columns else 'val_loss'
            values_by_loss[loss_key].extend(epoch_df['train_loss'].to_list())
            values_by_loss[loss_key].extend(epoch_df[val_metric].to_list())

    return {
        loss_key: [0, max(values) * padding]
        for loss_key, values in values_by_loss.items()
        if values
    }

def plot_training_curves(combined_df: pl.DataFrame, width: int = 280, height: int = 180,
                          y_domain: list = None, show_legend: bool = True) -> alt.LayerChart:
    if combined_df is None:
        return None

    val_metric = 'val_loss' if 'val_loss' in combined_df.columns else 'test_loss'
    n_runs = combined_df['run'].n_unique()

    # # Normalize column name so the legend always shows "val_loss"
    # if val_metric != 'test_loss':
    #     combined_df = combined_df.rename({val_metric: 'test_loss'})
    #     val_metric = 'test_loss'

    long_loss = combined_df.unpivot(on=['train_loss', val_metric], index=['epoch', 'run'])

    y_loss = alt.Y('value:Q', title='Loss',
                    scale=alt.Scale(domain=y_domain) if y_domain else alt.Undefined)
    color_loss = alt.Color('variable:N', title='Metric', scale=alt.Scale(scheme='category10'),
                            legend=alt.Legend() if show_legend else None)

    # thin, faint line per run (no legend entries thanks to `detail`)
    chart_runs = alt.Chart(long_loss).mark_line(opacity=0.25 if n_runs > 1 else 1.0, strokeWidth=1).encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=y_loss,
        color=color_loss,
        detail='run:N',
        tooltip=['epoch:Q', 'value:Q', 'variable:N', 'run:N'],
    )

    layers = [chart_runs]
    if n_runs > 1:
        chart_mean = alt.Chart(long_loss).mark_line(strokeWidth=2.5).encode(
            x='epoch:Q',
            y='mean(value):Q',
            color=color_loss,
        )
        layers.append(chart_mean)

    chart_loss = alt.layer(*layers).properties(title='Loss')

    lr_df = combined_df.unpivot(on=['learning_rate'], index=['epoch', 'run'])
    chart_lr = alt.Chart(lr_df).mark_line(point=False, strokeDash=[4, 4], opacity=0.5).encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=alt.Y('value:Q', axis=alt.Axis(title='Learning rate', orient='right')),
        color=alt.Color('variable:N', title='Learning Rate',
                         scale=alt.Scale(domain=['learning_rate'], range=['#bbbbbb']),
                         legend=alt.Legend() if show_legend else None),
        detail='run:N',
        tooltip=['epoch:Q', 'value:Q', 'run:N'],
    ).properties(title='Learning rate')

    combined = alt.layer(chart_loss, chart_lr).resolve_scale(
        y='independent', color='independent'
    ).properties(width=width, height=height, title='Loss and Learning Rate')

    return combined

In [ ]:
charts = []
summaries = []
per_run_summaries = {}  # optional: keep raw per-run data around too

use_shared_y_scale = True
y_domains_by_loss = compute_shared_y_domains_by_loss(model_paths) if use_shared_y_scale else {}


for i, (name, p) in enumerate(model_paths.items()):
    combined_df = load_all_runs(p)
    if combined_df is None:
        continue

    loss_key = name.rsplit("_", 1)[-1]
    y_domain = y_domains_by_loss.get(loss_key) if use_shared_y_scale else None

    show_legend = (i % 2 == 1)
    chart = plot_training_curves(combined_df, width=280, height=180, y_domain=y_domain, show_legend=show_legend)
    title = name.split('_')[0] + ' ' + name.split('_')[1] + ' | '+ ' | '.join(name.split('_')[2:])
    #title = ' | '.join(name.split('_'))
    
    chart = chart.properties(title=alt.TitleParams(title, fontSize=12, fontWeight="bold"))
    charts.append(chart)

    run_summaries = collect_run_summaries(p)
    per_run_summaries[name] = run_summaries
    agg = aggregate_summaries(run_summaries)
    summaries.append({"model": name, **agg})

In [ ]:
cols = 2
rows = [
    alt.hconcat(*charts[i: i + cols]).resolve_scale(color="independent")
    for i in range(0, len(charts), cols)
]

grid = (
    alt.vconcat(*rows)
    .configure_view(strokeWidth=0)
    .configure_title(anchor="start")
    .properties(title="Training curves – all models (mean ± individual runs)")
)
grid

In [ ]:
df = pd.DataFrame(summaries).set_index("model")

df.index = pd.MultiIndex.from_tuples(
    [tuple(n.rsplit("_", 1)) for n in df.index],
    names=["architecture", "loss"],
)

metrics = ["best_train_loss", "best_val_loss", "final_train_loss", "final_val_loss"]

def mean_std_fmt(mean, std):
    if pd.isna(mean):
        return "—"
    if pd.isna(std) or std == 0:
        return f"{mean:,.2f}"
    return f"{mean:,.2f} ± {std:,.2f}"

# Build display-only formatted columns, keep raw *_mean for gradient coloring
display_cols = {}
for m in metrics:
    mean_col, std_col = f"{m}_mean", f"{m}_std"
    if mean_col in df.columns:
        display_cols[m] = [
            mean_std_fmt(mean, std) for mean, std in zip(df[mean_col], df.get(std_col, [None]*len(df)))
        ]

display_df = df.copy()
for m, vals in display_cols.items():
    display_df[m] = vals  # overwrite with formatted string for display

show_cols = ["n_runs", "total_epochs_mean"] + metrics
gradient_cols = [f"{m}_mean" for m in ["best_train_loss", "best_val_loss"] if f"{m}_mean" in df.columns]

(
    display_df[show_cols]
    .style
    .format({"n_runs": "{:.0f}", "total_epochs_mean": "{:.1f}"})
    # apply gradient using the *raw* mean values, but they display formatted text below
    .background_gradient(subset=pd.IndexSlice[:, ["best_train_loss"]],
                          cmap="RdYlGn_r", gmap=df["best_train_loss_mean"])
    .background_gradient(subset=pd.IndexSlice[:, ["best_val_loss"]],
                          cmap="RdYlGn_r", gmap=df["best_val_loss_mean"])
    .set_caption("Model summary – mean ± std across runs (lower loss is better)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "6px 12px")]},
        {"selector": "td", "props": [("padding", "5px 12px")]},
    ])
)

In [ ]:
def compute_shared_y_domains_by_loss(training_variants: Dict, padding: float = 1.05) -> Dict:
    """Compute a shared loss y-domain per loss type across training variants.

    Returns a dict mapping loss class -> [0, max_value] domain.
    """
    values_by_loss = {}
    for item in training_variants.values():
        loss_key = item['loss']
        model_path = item['model_path']
        logger = TrainingLogger(log_dir=model_path)
        metrics = logger.load_metrics_file()
        epoch_df = metrics.get('epoch_metrics', None)
        if epoch_df is None:
            continue
        values_by_loss.setdefault(loss_key, [])
        values_by_loss[loss_key].extend(epoch_df['train_loss'].to_list())
        values_by_loss[loss_key].extend(epoch_df['test_loss'].to_list())

    return {
        loss_key: [0, max(values) * padding]
        for loss_key, values in values_by_loss.items()
        if values
    }


# toggle this to switch between shared (per loss type) and independent y-scales
use_shared_y_scale = True

y_domains_by_loss = compute_shared_y_domains_by_loss(training_variants) if use_shared_y_scale else {}

charts = []
summaries = []
for name, item in training_variants.items():
    model_path = item['model_path']
    logger = TrainingLogger(log_dir=model_path)
    metrics = logger.load_metrics_file()
    summary = logger.get_summary()

    y_domain = y_domains_by_loss.get(item['loss']) if use_shared_y_scale else None

    # Tag the chart with a title so each panel is self-labelled
    chart = plot_training_curves(metrics, y_domain=y_domain).properties(
        title=alt.TitleParams(name, fontSize=12, fontWeight="bold"),
        width=280,
        height=180,
    )
    charts.append(chart)
    summaries.append({"model": name, **summary})

In [7]:
for i, chart in enumerate(charts):
    orig_title = chart.title.text if chart.title else ""
    new_title = orig_title.split('_')[0] + ' ' + orig_title.split('_')[1] + ' | '+ ' | '.join(orig_title.split('_')[2:])
    if i%2 == 1:
        # Keep the legend
        charts[i] = chart.properties(title=new_title)
    else:
        # Hide the legend
        charts[i] = chart.encode(
            color=alt.Color("your_field:N", legend=None)
        ).properties(title=new_title)


# plots
cols = 2
rows = [
    alt.hconcat(*charts[i : i + cols], center=True).resolve_scale(color="shared")
    for i in range(0, len(charts), cols)
]

grid = (
    alt.vconcat(*rows,)
    .configure_view(strokeWidth=0)
    .configure_title(anchor="start")
    .properties(title="Training curves – all models")
)
grid

alt.VConcatChart(...)

In [8]:
df = pd.DataFrame(summaries).set_index("model")

# Split model name into architecture + loss for easier scanning
df.index = pd.MultiIndex.from_tuples(
    [tuple(n.rsplit("_", 1)) for n in df.index],
    names=["architecture", "loss"],
)

df_print = (
    df.style
    .format({
        "total_epochs": "{:.0f}",
        "best_train_loss": "{:,.2f}",
        "best_test_loss": "{:,.2f}",
        "final_train_loss": "{:,.2f}",
        "final_test_loss": "{:,.2f}",
    })
    .background_gradient(subset=["best_test_loss"], cmap="RdYlGn_r")
    .background_gradient(subset=["best_train_loss"], cmap="RdYlGn_r")
    .set_caption("Model summary – lower loss is better")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "6px 12px")]},
        {"selector": "td", "props": [("padding", "5px 12px")]},
    ])
)
df_print

,,total_epochs,best_train_loss,best_test_loss,final_train_loss,final_test_loss
architecture,loss,,,,,
custom_tirex_shift_slstm,quantile,100,74.78,559.20,74.78,689.76
custom_tirex_shift_lstm,quantile,100,41.41,571.46,41.41,834.38
custom_tirex_cpm_slstm,quantile,100,42.21,308.27,42.21,408.31
custom_tirex_cpm_lstm,quantile,100,29.07,322.94,29.07,420.40
custom_tirex_shift_slstm,mse,100,"2,901,915.03","4,634,443.72","2,901,915.03","4,664,941.92"
custom_tirex_shift_lstm,mse,100,"46,591.36","9,396,074.49","46,591.36","11,991,348.54"
custom_tirex_cpm_slstm,mse,100,"97,453.89","8,169,185.35","97,453.89","11,653,105.38"
custom_tirex_cpm_lstm,mse,100,"59,365.80","7,820,493.16","59,365.80","8,427,848.27"
custom_tirex_shift_slstm,l1,100,"1,246.28","1,534.37","1,246.28","1,608.44"


In [9]:
print(df_print.to_latex(hrules=True))

\begin{table}
\caption{Model summary – lower loss is better}
\thleft
\td5px 12px
\begin{tabular}{llrrrrr}
\toprule
 &  & total_epochs & best_train_loss & best_test_loss & final_train_loss & final_test_loss \\
architecture & loss &  &  &  &  &  \\
\midrule
custom_tirex_shift_slstm & quantile & 100 & \background-color#006837 \color#f1f1f1 74.78 & \background-color#006837 \color#f1f1f1 559.20 & 74.78 & 689.76 \\
custom_tirex_shift_lstm & quantile & 100 & \background-color#006837 \color#f1f1f1 41.41 & \background-color#006837 \color#f1f1f1 571.46 & 41.41 & 834.38 \\
custom_tirex_cpm_slstm & quantile & 100 & \background-color#006837 \color#f1f1f1 42.21 & \background-color#006837 \color#f1f1f1 308.27 & 42.21 & 408.31 \\
custom_tirex_cpm_lstm & quantile & 100 & \background-color#006837 \color#f1f1f1 29.07 & \background-color#006837 \color#f1f1f1 322.94 & 29.07 & 420.40 \\
custom_tirex_shift_slstm & mse & 100 & \background-color#a50026 \color#f1f1f1 2,901,915.03 & \background-color#fdfebc \col

## Evaluation

In [8]:
model_paths = {}
for name, item in training_variants.items():
    model_path = item['model_path']
    model_paths[name] = model_path
model_paths['original_pretrained'] = Path("../models/tirex_original_pretrained_20260614")
model_paths

{'custom_tirex_shift_slstm_quantile': WindowsPath('../models/custom_tirex_shift_slstm_quantile_20260630_232204'),
 'custom_tirex_shift_lstm_quantile': WindowsPath('../models/custom_tirex_shift_lstm_quantile_20260701_060043'),
 'custom_tirex_cpm_slstm_quantile': WindowsPath('../models/custom_tirex_cpm_slstm_quantile_20260701_072041'),
 'custom_tirex_cpm_lstm_quantile': WindowsPath('../models/custom_tirex_cpm_lstm_quantile_20260701_140218'),
 'custom_tirex_shift_slstm_mse': WindowsPath('../models/custom_tirex_shift_slstm_mse_20260701_233605'),
 'custom_tirex_shift_lstm_mse': WindowsPath('../models/custom_tirex_shift_lstm_mse_20260701_214605'),
 'custom_tirex_cpm_slstm_mse': WindowsPath('../models/custom_tirex_cpm_slstm_mse_20260701_235122'),
 'custom_tirex_cpm_lstm_mse': WindowsPath('../models/custom_tirex_cpm_lstm_mse_20260702_060352'),
 'custom_tirex_shift_slstm_l1': WindowsPath('../models/custom_tirex_shift_slstm_l1_20260702_070515'),
 'custom_tirex_shift_lstm_l1': WindowsPath('../mod

In [9]:
scores = []
for name, p in model_paths.items():
    with open(p / "scores.json", 'r') as f:
        s = json.load(f)
    scores.append({"model": name, **s})

SCORE_COLS = ["score_smape", "score_quantile"]

rows = []
for entry in scores:          # your list of dicts
    model = entry["model"]
    best = entry["last_model.pth"]
    for pred_mode, values in best.items():
        rows.append({
            "model": model,
            "pred_mode": pred_mode,
            **dict(zip(SCORE_COLS, values)),
        })

df_scores = pd.DataFrame(rows).set_index(["model", "pred_mode"])
df_flat = df_scores.unstack("pred_mode")

# Build gradient styles — lower=better for smape/mase/wis, higher=better for r2
lower_better = ["score_smape", "score_quantile"]
higher_better = []

df_style = (
    df_flat.style
    .format({
        "score_smape": "{:.4f}",
        #"score_mase":  "{:.4f}",
        #"score_r2":    "{:.4f}",
        "score_wis":   "{:,.1f}",
    })
    #.background_gradient(subset=lower_better,  cmap="RdYlGn_r")  # red=bad, green=good
    #.background_gradient(subset=higher_better, cmap="RdYlGn")    # green=good (no _r)
    .set_caption("Best model scores by model and prediction mode")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "6px 12px"), ("white-space", "nowrap")]},
        {"selector": "td", "props": [("padding", "5px 12px"), ("white-space", "nowrap")]},
    ]) 
) 
df_style

In [25]:
print(df_style.to_latex(hrules=True))

\begin{table}
\caption{Best model scores by model and prediction mode}
\thleft
\td5px 12px
\begin{tabular}{lrrrr}
\toprule
 & \multicolumn{2}{r}{score_smape} & \multicolumn{2}{r}{score_quantile} \\
pred_mode & autoregressive_mean & original & autoregressive_mean & original \\
model &  &  &  &  \\
\midrule
custom_tirex_cpm_lstm_l1 & 0.561026 & 0.549579 & 7.653984 & 9.129549 \\
custom_tirex_cpm_lstm_mse & 0.543577 & 0.561027 & 7.804971 & 9.666832 \\
custom_tirex_cpm_lstm_quantile & 0.682525 & 0.679872 & 11.472909 & 12.407183 \\
custom_tirex_cpm_slstm_l1 & 0.590527 & 0.569702 & 7.922503 & 8.954371 \\
custom_tirex_cpm_slstm_mse & 0.555255 & 0.572576 & 8.268623 & 9.205929 \\
custom_tirex_cpm_slstm_quantile & 0.663669 & 0.679483 & 11.578801 & 13.107755 \\
custom_tirex_shift_lstm_l1 & 0.669059 & 0.710374 & 12.041589 & 18.204909 \\
custom_tirex_shift_lstm_mse & 0.567039 & 0.558245 & 8.562264 & 8.638710 \\
custom_tirex_shift_lstm_quantile & 0.674492 & 0.686572 & 11.953254 & 12.777628 \\
custom_